In [2]:
import gzip
import pandas as pd
import numpy as np

# =============================================================================
# Step 1. Load Gene Expression Data
# =============================================================================

Gene_expression_Dataset_name = r"F:\GeneformerProject\data\EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena.gz"

with gzip.open(Gene_expression_Dataset_name, 'rt') as f:
    gene_expr = pd.read_csv(f, sep="\t", index_col=0)

print("="*60)
print("Gene Expression Dataset")
print("="*60)
print("Shape:", gene_expr.shape)
print("Rows    : Genes")
print("Columns : TCGA Samples of Patient")
print()

# =============================================================================
# Step 2. Load Clinical Data
# =============================================================================

Clinical_Dataset_name = r"F:\ML\data\TCGA_phenotype_denseDataOnlyDownload.tsv.gz"

with gzip.open(Clinical_Dataset_name, 'rt') as f:
    clinical = pd.read_csv(f, sep="\t", index_col=0)

print("="*60)
print("Clinical Dataset")
print("="*60)
print("Shape:", clinical.shape)
print()

print("="*60)
print("An example of gene expression:\n", gene_expr.head())
print("An example of related clinical data:\n", clinical.head())
#duplicate columns in gene_expr
print("number of duplicated gene expression=", gene_expr.columns.duplicated().sum())
print("number of duplicated clinical=",clinical.index.duplicated().sum())
#print gene name
print(gene_expr.index[:10])
# =============================================================================
# Step 3. Match Samples
# =============================================================================

common_samples = gene_expr.columns.intersection(clinical.index)
#print(common_samples)

print("="*60)
print("Sample Matching")
print("="*60)
print("Expression Samples :", len(gene_expr.columns))
print("Clinical Samples   :", len(clinical.index))
print("Matched Samples    :", len(common_samples))
print()

# Keep only common samples

gene_expr = gene_expr[common_samples]
clinical = clinical.loc[common_samples]

#check data type
print("gene expression data type=",gene_expr.dtypes.unique())
gene_expr = gene_expr.astype(np.float32)
# =============================================================================
# Step 4. Transpose Gene Matrix
# =============================================================================

gene_expr = gene_expr.T

print("="*60)
print("Transposed Gene Expression")
print("="*60)
print(gene_expr.shape)
print()

# =============================================================================
# Step 5. Explore Clinical Columns
# =============================================================================

print("="*60)
print("Clinical Columns")
print("="*60)

for c in clinical.columns:
    print(c)

print()

# =============================================================================


TARGET_COLUMN = "_primary_disease"   
TISSUE_COLUMN = "sample_type"
# =============================================================================
# Step 6. Build Final Dataset
# =============================================================================

dataset = gene_expr.copy()

dataset["Cancer_Type"] = clinical[TARGET_COLUMN]
dataset["tissue_type"] = clinical[TISSUE_COLUMN]

print("="*60)
print("Final Dataset")
print("="*60)
print(dataset.shape)
print()

print(dataset.head())

# =============================================================================
# Step 7. Remove Missing Labels
# =============================================================================

dataset = dataset.dropna(subset=["Cancer_Type"])

print("="*60)
print("After Removing Missing Cancer Labels")
print("="*60)
print(dataset.shape)
print()

# =============================================================================
# Step 8. Cancer Distribution
# =============================================================================

print("="*60)
print("Cancer Type Distribution")
print("="*60)

print(dataset["Cancer_Type"].value_counts())
print(dataset["Cancer_Type"].nunique())
print(dataset["Cancer_Type"].value_counts().describe())


# =============================================================================
# Step 9. Missing Values
# =============================================================================

print("="*60)
print("Missing Values")
print("="*60)

print("Total Missing:", dataset.isnull().sum().sum())
print(dataset.shape)

print(dataset["Cancer_Type"].isnull().sum())

missing_per_patient = dataset.isnull().sum(axis=1)
print(missing_per_patient.describe())

print("Patients with NO missing values:",
      (missing_per_patient == 0).sum())

print("Patients with missing values:",
      (missing_per_patient > 0).sum())
'''
for i in range(len(dataset)):
    if dataset.iloc[i].isnull().sum()>0:
        print(dataset.iloc[i])
'''
# =============================================================================
# Step 10. Save Dataset
# =============================================================================

dataset.to_csv("TCGA_Cancer_Dataset.csv")

print()
print("="*60)
print("Dataset saved as TCGA_Cancer_Dataset.csv")
print("="*60)



Gene Expression Dataset
Shape: (20531, 11069)
Rows    : Genes
Columns : TCGA Samples of Patient

Clinical Dataset
Shape: (12804, 3)

An example of gene expression:
            TCGA-OR-A5J1-01  TCGA-OR-A5J2-01  TCGA-OR-A5J3-01  TCGA-OR-A5J5-01  \
sample                                                                          
100130426             0.00             0.00             0.00             0.00   
100133144             2.09             1.88             1.45             0.00   
100134869             2.30             3.32             2.92             1.35   
10357                 7.23             6.36             6.45             5.78   
10431                10.99            10.35            10.04            11.20   

           TCGA-OR-A5J6-01  TCGA-OR-A5J7-01  TCGA-OR-A5J8-01  TCGA-OR-A5J9-01  \
sample                                                                          
100130426             0.00             0.00             0.00             0.00   
100133144             0.